# AI Tutor on Kaggle

Enable a GPU accelerator and Internet, configure the next cell, then use **Run All**. State restoration is attempted only once per Kaggle runtime, so a second Run All cannot overwrite newer active-session data.

In [ ]:
# User configuration: edit values only in this cell.
REPOSITORY_URL = "https://github.com/qtrung123/AI_Tutor2-Kaggle.git"
REPOSITORY_BRANCH = "main"
PROJECT_ROOT = "/kaggle/working/AI_Tutor2"
UPDATE_CODE = False  # Set True for one run when you intentionally want to pull updates.
QUICK_START = True   # Skips tests and unchanged pip/npm installs.
GGUF_MODEL_PATH = "/kaggle/input/datasets/trung121212/tutor-model/qwen2.5-7b-instruct-q4_k_m.gguf"
LECTURE_INPUT_DIR = ""
OLLAMA_CHAT_MODEL = "qwen-tutor-7b"
OLLAMA_EMBEDDING_MODEL = "bge-m3"
BACKEND_PORT = 8000
FRONTEND_PORT = 3000
PUBLIC_PORT = 7860
TUNNEL_PROVIDER = "ngrok"
RECREATE_OLLAMA_MODEL = False
STATE_DATASET_SLUG = ""
CACHE_DATASET_SLUG = ""

import os
for key, value in {
    "PROJECT_ROOT": PROJECT_ROOT, "GGUF_MODEL_PATH": GGUF_MODEL_PATH,
    "OLLAMA_CHAT_MODEL": OLLAMA_CHAT_MODEL, "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "BACKEND_PORT": BACKEND_PORT, "FRONTEND_PORT": FRONTEND_PORT, "PUBLIC_PORT": PUBLIC_PORT,
    "RECREATE_OLLAMA_MODEL": int(RECREATE_OLLAMA_MODEL), "REBUILD_CHROMA_ON_EMBEDDING_CHANGE": 1,
    "QUIZ_GENERATION_RETRY_LIMIT": 3,
}.items(): os.environ[key] = str(value)

In [ ]:
from pathlib import Path
import subprocess
root = Path(PROJECT_ROOT)
if not (root / ".git").exists():
    root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, PROJECT_ROOT], check=True)
elif UPDATE_CODE:
    subprocess.run(["git", "-C", PROJECT_ROOT, "fetch", "origin", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "checkout", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", REPOSITORY_BRANCH], check=True)
else:
    print("Using existing checkout (UPDATE_CODE=False).")

## Restore state and build cache once per runtime

In [ ]:
import hashlib, os, shutil, sys
from pathlib import Path
sys.path.insert(0, f"{PROJECT_ROOT}/deployment")
import kaggle_persist

def _load_kaggle_credentials():
    if os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"): return True
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
        return True
    except Exception as error:
        print("Kaggle persistence credentials unavailable:", error)
        return False

STAGING_DIR = Path("/kaggle/working/.persist")
CACHE_DIR = Path("/kaggle/working/.cache")
RUNTIME_MARKERS = Path("/kaggle/working/.ai-tutor-runtime-markers")
for directory in (STAGING_DIR, CACHE_DIR, RUNTIME_MARKERS): directory.mkdir(parents=True, exist_ok=True)
PERSISTENCE_ENABLED = bool(STATE_DATASET_SLUG) and _load_kaggle_credentials()
CACHE_ENABLED = bool(CACHE_DATASET_SLUG) and _load_kaggle_credentials()

def _restore_once(kind, slug, enabled, names, destination):
    marker_key = hashlib.sha256(f"{kind}:{slug}".encode()).hexdigest()[:16]
    marker = RUNTIME_MARKERS / f"{kind}-restore-attempted-{marker_key}"
    if marker.exists():
        print(f"Skipping {kind} restore: already attempted in this runtime.")
        return
    marker.touch()  # Guard before remote access: Run All can never restore this snapshot twice.
    if not enabled:
        print(f"{kind.title()} persistence disabled.")
        return
    restore_dir = STAGING_DIR / f"{kind}_restore"
    if not kaggle_persist.restore_dataset(slug, restore_dir):
        print(f"No saved {kind} snapshot found.")
        return
    for name in names:
        source = restore_dir / name
        if not source.exists(): continue
        target = destination / name
        if target.exists(): shutil.rmtree(target) if target.is_dir() else target.unlink()
        shutil.move(str(source), str(target))
    print(f"Restored {kind} snapshot once for this runtime.")

_restore_once("state", STATE_DATASET_SLUG, PERSISTENCE_ENABLED, ("data", "vectorstore"), Path(PROJECT_ROOT))
_restore_once("cache", CACHE_DATASET_SLUG, CACHE_ENABLED, ("ollama-models", "pip", "npm"), CACHE_DIR)
for name in ("ollama-models", "pip", "npm"): (CACHE_DIR / name).mkdir(parents=True, exist_ok=True)
os.environ["OLLAMA_MODELS"] = str(CACHE_DIR / "ollama-models")
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["NPM_CONFIG_CACHE"] = str(CACHE_DIR / "npm")

In [ ]:
import shutil, subprocess
required = [("nginx", "nginx"), ("curl", "curl"), ("zstd", "zstd")]
missing = [package for package, binary in required if not shutil.which(binary)]
if missing:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.run(["ollama", "--version"], check=True)

In [ ]:
import hashlib, subprocess, sys
setup_markers = RUNTIME_MARKERS / "setup"
setup_markers.mkdir(exist_ok=True)
requirements = Path(PROJECT_ROOT) / "requirements.txt"
requirements_key = hashlib.sha256(requirements.read_bytes()).hexdigest()
pip_marker = setup_markers / f"pip-{requirements_key}"
if QUICK_START and pip_marker.exists(): print("Skipping unchanged pip requirements.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
    pip_marker.touch()
lockfile = Path(PROJECT_ROOT) / "frontend/package-lock.json"
npm_key = hashlib.sha256(lockfile.read_bytes()).hexdigest()
npm_marker = setup_markers / f"npm-{npm_key}"
if QUICK_START and npm_marker.exists(): print("Skipping unchanged npm dependencies.")
else:
    subprocess.run(["npm", "ci", "--prefer-offline"], cwd=lockfile.parent, check=True)
    npm_marker.touch()

In [ ]:
from pathlib import Path
import shutil
data_dir = Path(PROJECT_ROOT) / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(Path(PROJECT_ROOT) / "vectorstore").mkdir(parents=True, exist_ok=True)
if LECTURE_INPUT_DIR:
    source_dir = Path(LECTURE_INPUT_DIR)
    if not source_dir.is_dir(): raise FileNotFoundError(f"Lecture input not found: {source_dir}")
    for source in source_dir.rglob("*"):
        if source.is_file() and source.suffix.lower() in {".pdf", ".txt"}:
            target = data_dir / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size: shutil.copy2(source, target)
print("Runtime data directory:", data_dir)

In [ ]:
import subprocess, sys
if QUICK_START:
    print("QUICK_START=True: skipping pytest.")
else:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_ROOT, check=True)

In [ ]:
import subprocess
subprocess.run(["bash", f"{PROJECT_ROOT}/deployment/start_kaggle.sh"], cwd=PROJECT_ROOT, check=True)
print("Services restarted and validated by start_kaggle.sh.")

In [ ]:
import subprocess, sys
if TUNNEL_PROVIDER != "ngrok": print(f"Unsupported tunnel provider: {TUNNEL_PROVIDER}")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
        try: from pyngrok import ngrok
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
            from pyngrok import ngrok
        ngrok.kill(); ngrok.set_auth_token(token)
        tunnel = ngrok.connect(addr=f"127.0.0.1:{PUBLIC_PORT}", proto="http")
        print(f"AI Tutor URL: {tunnel.public_url}")
    except Exception as error: print("Could not create tunnel:", error)

## Save progress
Set `SAVE_STATE_NOW=True` before ending a session. SQLite state is inside `data/`; `indexed_files.json` is not persisted.

In [ ]:
SAVE_STATE_NOW = False
if not SAVE_STATE_NOW: print("Nothing saved. Set SAVE_STATE_NOW=True and rerun this cell.")
elif not PERSISTENCE_ENABLED: print("State persistence is disabled.")
else:
    staging = STAGING_DIR / "state_save"
    if staging.exists(): shutil.rmtree(staging)
    staging.mkdir(parents=True)
    for name in ("data", "vectorstore"):
        source = Path(PROJECT_ROOT) / name
        if source.exists(): shutil.copytree(source, staging / name)
    kaggle_persist.save_dataset(STATE_DATASET_SLUG, staging, title="AI Tutor state", message="Active-session snapshot")
    print("Progress saved.")

## Save cache (optional)

In [ ]:
SAVE_CACHE_NOW = False
if not SAVE_CACHE_NOW: print("Nothing saved. Set SAVE_CACHE_NOW=True and rerun this cell.")
elif not CACHE_ENABLED: print("Cache persistence is disabled.")
else:
    staging = STAGING_DIR / "cache_save"
    if staging.exists(): shutil.rmtree(staging)
    staging.mkdir(parents=True)
    for name in ("ollama-models", "pip", "npm"):
        source = CACHE_DIR / name
        if source.exists() and any(source.iterdir()): shutil.copytree(source, staging / name)
    kaggle_persist.save_dataset(CACHE_DATASET_SLUG, staging, title="AI Tutor build cache", message="Updated runtime cache")
    print("Cache saved.")